# TD 8 — Prix de l'immobilier et effets fixes

**Analyse des données — L3 Économie**

Les **DVF** recensent toutes les ventes immobilières de France, avec leur prix. Vous allez estimer une **régression hédonique** — le prix expliqué par les caractéristiques du bien — puis y ajouter des **effets fixes** géographiques, et regarder les coefficients se déplacer.

Trois départements : le Val-d'Oise (95), les Yvelines (78) et l'Oise (60).

> *Exécution → Tout exécuter* avant de démarrer. Le chargement prend une minute.


## Partie 1 — Ce que décrit une ligne

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

BASE = "https://files.data.gouv.fr/geo-dvf/latest/csv/{annee}/departements/{dep}.csv.gz"
ANNEE = 2024
DEPTS = ["95", "78", "60"]          # Val-d'Oise, Yvelines, Oise

# D'abord : que contient ce fichier ? On lit cinq lignes avant de tout charger.
apercu = pd.read_csv(BASE.format(annee=ANNEE, dep="95"), nrows=5)
print(len(apercu.columns), "colonnes\n")
print(apercu.columns.tolist())

In [ ]:
COLS = ["id_mutation", "date_mutation", "nature_mutation", "valeur_fonciere",
        "code_commune", "nom_commune", "code_departement",
        "type_local", "surface_reelle_bati", "nombre_pieces_principales",
        "surface_terrain"]

morceaux = []
for dep in DEPTS:
    m = pd.read_csv(BASE.format(annee=ANNEE, dep=dep), usecols=COLS,
                    dtype={"code_commune": str, "code_departement": str})
    print(dep, ":", len(m), "lignes")
    morceaux.append(m)

df = pd.concat(morceaux, ignore_index=True)
print("\ntotal :", len(df), "lignes |", df["id_mutation"].nunique(), "mutations")
df.head(3)

**Question 1.** Combien de lignes, combien de mutations distinctes ? Que vous apprend l'écart ?

*Votre réponse :*


In [ ]:
# une mutation qui compte plusieurs lignes
gros = df["id_mutation"].value_counts()
exemple = gros.index[0]

df[df["id_mutation"] == exemple][["id_mutation", "valeur_fonciere", "type_local",
                                  "surface_reelle_bati", "surface_terrain"]]

**Question 2.** Que vaut `valeur_fonciere` sur chaque ligne de cette mutation ? Qu'est-ce qu'une ligne décrit ?

**Question 3.** Comparez la somme naïve et la somme correcte.

*Vos réponses :*


In [ ]:
# a vous : les deux sommes, et l'ecart en %

**Question 4.** Construisez le fichier d'analyse, en comptant les pertes à chaque étape.

In [ ]:
# une ligne = un logement vendu seul
lgt = df[df["type_local"].isin(["Appartement", "Maison"])].copy()
print("lignes de logement        :", len(lgt))

taille = lgt.groupby("id_mutation")["id_mutation"].transform("size")
d = lgt[taille == 1].copy()
print("mutations a un seul local :", len(d))

d = d[d["nature_mutation"] == "Vente"]
print("ventes uniquement         :", len(d))

d = d.dropna(subset=["valeur_fonciere", "surface_reelle_bati"])
d = d[d["surface_reelle_bati"] >= 9]
print("surface renseignee >= 9 m2:", len(d))

*Combien de lignes à chaque étape, et pourquoi ?*

## Partie 2 — La régression hédonique

**Question 5.** Le prix au m², ses valeurs extrêmes, et vos bornes.


In [ ]:
d["prix_m2"] = d["valeur_fonciere"] / d["surface_reelle_bati"]
print("avant bornes : min", round(d["prix_m2"].min(), 2),
      "| median", round(d["prix_m2"].median()),
      "| max", round(d["prix_m2"].max()))

d = d[(d["prix_m2"] > 300) & (d["prix_m2"] < 20000)].copy()
print("apres bornes :", len(d), "logements |", d["code_commune"].nunique(), "communes")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(d["prix_m2"], bins=60, color="#9AA0A0")
ax.set_xlabel("Prix au mètre carré (euros)")
ax.set_ylabel("Nombre de ventes")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
d["lprix"]  = np.log(d["valeur_fonciere"])
d["lsurf"]  = np.log(d["surface_reelle_bati"])
d["maison"] = (d["type_local"] == "Maison").astype(int)
d["pieces"] = d["nombre_pieces_principales"]
d = d.dropna(subset=["lprix", "lsurf", "pieces"])
print(len(d), "observations pour l'estimation")

**Question 6.** Estimez `lprix ~ lsurf`. Énoncez l'élasticité en une phrase. Pourquoi ne vaut-elle pas 1 ?

*Votre réponse :*


In [ ]:
m1 = smf.ols("lprix ~ lsurf", data=d).fit(cov_type="HC1")
print(m1.summary().tables[1])

**Question 7.** Ajoutez les pièces et la muette maison. Le coefficient de surface bouge-t-il ?

*Votre réponse :*


In [ ]:
# a vous

## Partie 3 — Les effets fixes

**Questions 8 et 9.** Ajoutez les effets fixes départementaux, puis communaux, et comparez.


In [ ]:
SPECS = {
 "M1 surface seule"     : "lprix ~ lsurf",
 "M2 + pieces + maison" : "lprix ~ lsurf + pieces + maison",
 "M3 + EF departement"  : "lprix ~ lsurf + pieces + maison + C(code_departement)",
 "M4 + EF commune"      : "lprix ~ lsurf + pieces + maison + C(code_commune)",
}

res = {}
for nom, f in SPECS.items():
    res[nom] = smf.ols(f, data=d).fit(cov_type="HC1")

def ligne(m):
    out = {v: round(m.params[v], 4) if v in m.params else None
           for v in ["lsurf", "pieces", "maison"]}
    out["R2"] = round(m.rsquared, 4)
    out["parametres"] = len(m.params)
    return out

tab = pd.DataFrame({nom: ligne(m) for nom, m in res.items()})
print(tab.to_string())

*Vos réponses 8 et 9 :*

**Question 10.** Pourquoi le coefficient de `maison` bouge-t-il autant ?


In [ ]:
# la part de maisons est-elle liee au niveau de prix de la commune ?
par_commune = d.groupby("code_commune").agg(part_maison=("maison", "mean"),
                                            prix_m2=("prix_m2", "mean"),
                                            ventes=("maison", "size"))
par_commune = par_commune[par_commune["ventes"] >= 30]
print("correlation :", round(par_commune["part_maison"].corr(par_commune["prix_m2"]), 3))

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.scatter(par_commune["part_maison"], par_commune["prix_m2"], s=18,
           alpha=.6, color="#3D4543")
ax.set_xlabel("Part de maisons dans les ventes de la commune")
ax.set_ylabel("Prix moyen au m² (euros)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

*Votre réponse 10 :*

**Question 11.** Quelles variables deviennent impossibles à estimer ? Deux exemples.

**Question 12.** Que les effets fixes communaux n'absorbent-ils pas ? Quel modèle publiez-vous ?

*Vos réponses :*


## Pour ceux qui ont terminé

**Question 13.** Écarts-types groupés par commune.

**Question 14.** Le classement des communes par effet fixe.

**Question 15.** Effets fixes commune × trimestre.


In [ ]:
# a vous